In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!head /kaggle/input/datasets/jakenty1/vnmathproblems2026/problems.csv -n 1

# LLM Programming Experiment on Kaggle

This notebook runs Ollama (Llama 3.1, Gemma 2, Qwen 2.5) on a Kaggle GPU.

**Kaggle Setup:**
1. **Settings**: Enable **Internet**.
2. **Accelerator**: Select **GPU T4 x2** or **GPU P100**.
3. **Input**: Add the dataset `jakenty1/vnmathproblems2026`.

**Robustness features:**
- `exec()` sandboxed with `EXEC_TIMEOUT_SEC` per test — infinite loops can no longer hang the notebook
- Ollama inference errors are logged and skipped (task recorded so it won't be retried)
- Results flushed to disk via `fsync` after every task
- Periodic `.backup` snapshots every `CHECKPOINT_EVERY` tasks
- Auto-stops at `MAX_RUNTIME_HOURS` so Kaggle saves outputs cleanly
- Resume: already-completed task IDs are skipped automatically


In [ ]:
!pip install zstandard zstd -q

In [ ]:
!sudo apt-get install -y zstd -q

In [ ]:
import os
import subprocess
import time
import requests
import json
import re
import ast
import csv
import signal
import shutil
from datetime import datetime

print("Installing Ollama via official script...")
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
print("Starting Ollama server...")
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"

subprocess.Popen(["ollama", "serve"],
                 stdout=open("ollama_stdout.log", "w"),
                 stderr=open("ollama_stderr.log", "w"))

print("Waiting for Ollama server to respond...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"Ollama server ready after {i+1}s")
            break
    except:
        pass
    time.sleep(1)
else:
    print("ERROR: Ollama server failed to start.")
    !tail -n 20 ollama_stderr.log

In [ ]:
!ollama list

## 2. Configuration

In [ ]:
# ── User config ───────────────────────────────────────────────────────────────
model_choice      = "gemma2:9b" #"llama3.1:8b"   # or "gemma2:9b" / "qwen2.5:7b"
start_problem     = 14
end_problem       = 50

MAX_RUNTIME_HOURS = 8.5             # stop before Kaggle's 9h kill
CHECKPOINT_EVERY  = 10             # backup every N completed tasks
EXEC_TIMEOUT_SEC  = 10             # max seconds per test case in exec()
# ─────────────────────────────────────────────────────────────────────────────

csv_path = "/kaggle/input/datasets/jakenty1/vnmathproblems2026/problems.csv"
if not os.path.exists(csv_path):
    csv_path = "/kaggle/input/vnmathproblems2026/problems.csv"

print(f"Pulling model {model_choice}...")
!ollama pull {model_choice}

## 3. Helper Functions

In [ ]:
def call_ollama(model, system_prompt, user_prompt):
    start = time.time()
    try:
        payload = {
            "model": model,
            "prompt": user_prompt,
            "stream": False,
            "options": {"temperature": 0.2, "num_predict": 2048}
        }
        if system_prompt:
            payload["system"] = system_prompt
        r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=300)
        r.raise_for_status()
        data = r.json()
        return data.get("response", ""), {
            "duration": time.time() - start,
            "prompt_tokens": data.get("prompt_eval_count", 0),
            "completion_tokens": data.get("eval_count", 0),
            "success": True, "error": ""
        }
    except Exception as e:
        return "", {
            "duration": time.time() - start,
            "prompt_tokens": 0, "completion_tokens": 0,
            "success": False, "error": str(e)
        }


def is_python_code(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False


def extract_python_code(response: str) -> str:
    matches = re.findall(r"```python\s*\n(.*?)```", response, re.DOTALL)
    if matches:
        return matches[-1].strip()
    matches = re.findall(r"```\s*\n(.*?)```", response, re.DOTALL)
    if matches:
        return matches[-1].strip()
    return response.strip()


# ── exec() with timeout — prevents infinite loops from hanging the notebook ───
class ExecTimeoutError(Exception):
    pass


def _timeout_handler(signum, frame):
    raise ExecTimeoutError("exec timed out")


def evaluate_code(code, test_list):
    if not code:
        return {
            "passed": False, "tests_passed": 0,
            "tests_total": len(test_list), "errors": ["No code extracted"]
        }

    if not is_python_code(code):
        return {
            "passed": False, "tests_passed": 0,
            "tests_total": len(test_list), "errors": ["Response is not valid Python"]
        }

    passed_count, errors = 0, []
    for test in test_list:
        namespace = {}
        try:
            signal.signal(signal.SIGALRM, _timeout_handler)
            signal.alarm(EXEC_TIMEOUT_SEC)   # start countdown
            exec(code, namespace, namespace)
            exec(test, namespace, namespace)
            signal.alarm(0)                  # cancel on success
            passed_count += 1
        except ExecTimeoutError:
            signal.alarm(0)
            errors.append(f"Test timed out (>{EXEC_TIMEOUT_SEC}s): {test}")
        except Exception as e:
            signal.alarm(0)
            errors.append(f"Test failed: {test} | Error: {str(e)}")
    return {
        "passed": passed_count == len(test_list) and len(test_list) > 0,
        "tests_passed": passed_count,
        "tests_total": len(test_list),
        "errors": errors
    }


def parse_csv_row(row):
    standard_tests  = json.loads(row.get('test_cases', '[]'))
    challenge_tests = json.loads(row.get('challenge_test_list', '[]'))
    return {
        "task_id": f"{row['task_id']}_{row['category_id']}",
        "description": row['prompt'].strip(),
        "test_list": standard_tests + challenge_tests
    }


def get_completed_ids(filename):
    if not os.path.exists(filename):
        return set()
    completed = set()
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            try:
                completed.add(json.loads(line).get("task_id"))
            except:
                continue
    return completed


def append_result(filename, result):
    """Append one result and fsync so data survives a crash."""
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def save_backup(src, label=""):
    if not os.path.exists(src):
        return
    ts  = datetime.now().strftime("%H%M%S")
    tag = f"_{label}" if label else ""
    dst = src.replace(".jsonl", f"{tag}_{ts}.backup.jsonl")
    shutil.copy2(src, dst)
    print(f"  [checkpoint] {os.path.basename(dst)}  ({os.path.getsize(dst):,} bytes)", flush=True)


def summarize_results(filename):
    if not os.path.exists(filename):
        print("No results file found.")
        return
    total = passed = 0
    with open(filename) as f:
        for line in f:
            try:
                d = json.loads(line)
                total  += 1
                passed += int(d.get("passed", False))
            except:
                continue
    print(f"\n{'='*40}")
    print(f"Results: {passed}/{total} tasks passed ({100*passed/total:.1f}%)" if total else "No results yet.")
    print(f"Output : {filename}")
    print(f"{'='*40}")


## 4. Main Evaluation Loop

In [ ]:
SESSION_START = time.time()
MAX_SECONDS   = MAX_RUNTIME_HOURS * 3600

output_filename = f"/kaggle/working/{model_choice.replace(':', '-')}_{start_problem}_{end_problem}.jsonl"
completed_ids   = get_completed_ids(output_filename)

problems_to_test = []
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            if start_problem <= int(row['task_id']) <= end_problem:
                prob = parse_csv_row(row)
                if prob['task_id'] not in completed_ids:
                    problems_to_test.append(prob)
        except:
            continue

print(f"Loaded    : {len(problems_to_test)} remaining tasks (range {start_problem}–{end_problem})")
print(f"Completed : {len(completed_ids)} tasks already done (will be skipped)")
print(f"Time limit: {MAX_RUNTIME_HOURS}h — loop self-stops before Kaggle kills the session")
print(f"Exec limit: {EXEC_TIMEOUT_SEC}s per test case — infinite loops will be killed")
print()

tasks_done   = 0
time_stopped = False

for prob in problems_to_test:
    # ── Time-guard ────────────────────────────────────────────────────────
    elapsed = time.time() - SESSION_START
    if elapsed >= MAX_SECONDS:
        print(f"\n[TIME LIMIT] Reached {MAX_RUNTIME_HOURS}h — stopping to let Kaggle save outputs.", flush=True)
        save_backup(output_filename, label="timelimit")
        time_stopped = True
        break

    # ── Inference ─────────────────────────────────────────────────────────
    elapsed_str = f"{elapsed/3600:.1f}h"
    print(f"[{elapsed_str}] Testing {prob['task_id']}...", end=" ", flush=True)

    response_text, meta = call_ollama(model_choice, None, prob['description'])
    if not meta['success']:
        print(f"SKIP (inference error: {meta['error']})", flush=True)
        # Record the failure so this task won't be retried on resume
        append_result(output_filename, {
            "timestamp": datetime.now().isoformat(), "model": model_choice,
            "task_id": prob['task_id'], "passed": False,
            "tests_passed": 0, "tests_total": len(prob['test_list']),
            "errors": [f"Inference error: {meta['error']}"],
            "duration_seconds": meta['duration'],
            "prompt_tokens": 0, "completion_tokens": 0,
            "response_text": "", "extracted_code": ""
        })
        tasks_done += 1
        continue

    extracted = extract_python_code(response_text)
    eval_res  = evaluate_code(extracted, prob['test_list'])
    status    = 'PASS' if eval_res['passed'] else 'FAIL'
    print(f"{status} ({eval_res['tests_passed']}/{eval_res['tests_total']})", flush=True)

    # ── Persist ───────────────────────────────────────────────────────────
    append_result(output_filename, {
        "timestamp":         datetime.now().isoformat(),
        "model":             model_choice,
        "task_id":           prob['task_id'],
        "passed":            eval_res['passed'],
        "tests_passed":      eval_res['tests_passed'],
        "tests_total":       eval_res['tests_total'],
        "errors":            eval_res['errors'],
        "duration_seconds":  meta['duration'],
        "prompt_tokens":     meta['prompt_tokens'],
        "completion_tokens": meta['completion_tokens'],
        "response_text":     response_text,
        "extracted_code":    extracted
    })
    tasks_done += 1

    # ── Periodic backup ───────────────────────────────────────────────────
    if tasks_done % CHECKPOINT_EVERY == 0:
        save_backup(output_filename, label=f"step{tasks_done}")

# ── Done ──────────────────────────────────────────────────────────────────
if not time_stopped:
    print("\nAll tasks completed!")
    save_backup(output_filename, label="final")

summarize_results(output_filename)

## 5. Run Additional Ranges

Edit `start_problem` / `end_problem` in the config cell and re-run from **Section 2** onwards.
Already-completed task IDs are skipped automatically via `get_completed_ids()`.

In [ ]:
# Quick inspection — run anytime to check current progress
summarize_results(output_filename)